In [ ]:
# 1. 환경 설정 (Drive 마운트 + 패키지 설치 + 캐시를 Drive로)
import os, subprocess, sys, shutil

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")
CACHE_DIR  = os.path.join(DRIVE_DIR, ".cache")

for d in [
    DRIVE_DIR,
    os.path.join(DRIVE_DIR, "checkpoints"),
    os.path.join(DRIVE_DIR, "submissions"),
    os.path.join(DRIVE_DIR, "data"),
    CACHE_DIR,
    WORK_DIR,
]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_DIR, "huggingface")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_DIR, "torch")
os.environ["XDG_CACHE_HOME"] = CACHE_DIR
os.environ["PIP_CACHE_DIR"] = os.path.join(CACHE_DIR, "pip")
os.environ["TMPDIR"] = os.path.join(CACHE_DIR, "tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

import torch
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({vram:.1f} GB)")
else:
    print("GPU 없음! 런타임 > 런타임 유형 변경 > GPU 선택")

free = shutil.disk_usage('/content').free / 1e9
print(f"로컬 디스크 여유: {free:.1f} GB")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm>=1.0.20", "albumentations>=1.4.0",
    "opencv-python-headless", "scikit-learn", "pandas", "tqdm",
])
print("패키지 설치 완료")

In [ ]:
# 2. 소스 파일 복사 + 심볼릭 링크 (체크포인트 -> Drive)
import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/dacon"
WORK_DIR  = "/content/dacon"

SRC_FILES = ["models.py", "datasets.py", "train.py", "inference.py"]
missing = []
for f in SRC_FILES:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join(WORK_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  {f}")
    else:
        missing.append(f)
        print(f"  [MISSING] {f}")

if missing:
    print(f"\n다음 파일을 Drive에 업로드하세요: {DRIVE_DIR}/")
    for f in missing:
        print(f"  - {f}")

for name in ["checkpoints", "submissions"]:
    link_path  = os.path.join(WORK_DIR, name)
    drive_path = os.path.join(DRIVE_DIR, name)
    os.makedirs(drive_path, exist_ok=True)

    if os.path.islink(link_path):
        os.unlink(link_path)
    elif os.path.isdir(link_path):
        shutil.rmtree(link_path)

    os.symlink(drive_path, link_path)
    print(f"  {name}/ -> Drive (영구 저장)")

ckpt_dir = os.path.join(DRIVE_DIR, "checkpoints")
ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pth')]
if ckpts:
    print(f"\nDrive 체크포인트 {len(ckpts)}개 발견 (--resume 으로 이어하기):")
    for c in sorted(ckpts):
        mb = os.path.getsize(os.path.join(ckpt_dir, c)) / 1e6
        tag = "[resume]" if "ckpt" in c else "[best]"
        print(f"  {tag} {c} ({mb:.0f} MB)")
else:
    print("\n체크포인트 없음 — 새로 학습 시작")

In [ ]:
# 2.5 코랩 경쟁 모드 패치 (로컬 파일 영향 없음)
# 더 강한 증강/TTA를 코랩 런타임에만 적용

from pathlib import Path

path = Path("/content/dacon/datasets.py")
text = path.read_text(encoding="utf-8")

old_train = '''def get_train_transforms(img_size=384):
    """경쟁급 최대 증강"""
    return A.Compose([
        A.RandomResizedCrop(
            (img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.05, 0.05), scale=(0.93, 1.07),
            rotate=(-10, 10), p=0.5),
        A.Perspective(scale=(0.02, 0.06), p=0.3),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.HueSaturationValue(
                hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=1.0),
            A.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=1.0),
        ], p=0.8),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
        ], p=0.2),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.10), p=1.0),
            A.ISONoise(p=1.0),
        ], p=0.2),
        A.RandomShadow(p=0.15),
        A.RandomFog(fog_coef_range=(0.1, 0.35), p=0.1),
        A.CoarseDropout(
            num_holes_range=(1, 6),
            hole_height_range=(16, 40), hole_width_range=(16, 40), p=0.25),
        A.ToGray(p=0.05),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])'''

new_train = '''def get_train_transforms(img_size=384):
    """코랩 경쟁 모드용 강화 증강"""
    return A.Compose([
        A.RandomResizedCrop(
            (img_size, img_size), scale=(0.72, 1.0), ratio=(0.88, 1.12), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.08, 0.08), scale=(0.90, 1.10),
            rotate=(-12, 12), p=0.6),
        A.Perspective(scale=(0.03, 0.08), p=0.35),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.25, contrast_limit=0.25, p=1.0),
            A.HueSaturationValue(
                hue_shift_limit=12, sat_shift_limit=18, val_shift_limit=18, p=1.0),
            A.ColorJitter(
                brightness=0.25, contrast=0.25, saturation=0.20, hue=0.06, p=1.0),
            A.CLAHE(clip_limit=(1, 4), p=1.0),
        ], p=0.9),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.1), p=1.0),
        ], p=0.25),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.12), p=1.0),
            A.ISONoise(p=1.0),
            A.ImageCompression(quality_range=(45, 90), p=1.0),
        ], p=0.25),
        A.RandomShadow(p=0.20),
        A.RandomFog(fog_coef_range=(0.1, 0.35), p=0.12),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(12, 56), hole_width_range=(12, 56), p=0.35),
        A.ToGray(p=0.08),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])'''

old_tta = '''def get_tta_transforms(img_size=384):
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    return [
        # 0: base
        A.Compose([A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        # 1: hflip
        A.Compose([A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        # 2: brightness
        A.Compose([A.Resize(img_size, img_size),
                    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        # 3: center crop
        A.Compose([A.CenterCrop(int(img_size * 0.9), int(img_size * 0.9)),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
    ]'''

new_tta = '''def get_tta_transforms(img_size=384):
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    return [
        A.Compose([A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size),
                    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(int(img_size * 0.9), int(img_size * 0.9)),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size),
                    A.CLAHE(clip_limit=(1, 3), p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.Resize(img_size, img_size),
                    A.Affine(scale=(0.97, 1.03), translate_percent=(-0.02, 0.02), rotate=(-3, 3), p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
    ]'''

if old_train in text:
    text = text.replace(old_train, new_train)
else:
    print('[WARN] train transform block not matched')

if old_tta in text:
    text = text.replace(old_tta, new_tta)
else:
    print('[WARN] tta transform block not matched')

path.write_text(text, encoding='utf-8')
print('코랩 런타임용 datasets.py 패치 완료')
print('  - train augmentation 강화')
print('  - TTA 4개 -> 6개 확대')

In [ ]:
# 3. 데이터 연결 (로컬 압축해제 대신 Drive data/ 직접 링크)
import os, shutil, zipfile
import pandas as pd

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")
DRIVE_DATA = os.path.join(DRIVE_DIR, "data")

if os.path.islink(LOCAL_DATA):
    os.unlink(LOCAL_DATA)
elif os.path.isdir(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)

if os.path.exists(DRIVE_DATA):
    os.symlink(DRIVE_DATA, LOCAL_DATA)
    print("data/ -> Drive 심볼릭 링크")
else:
    zip_path = os.path.join(DRIVE_DIR, "data.zip")
    if os.path.exists(zip_path):
        print("Drive에 data.zip만 있고 data/ 폴더가 없습니다.")
        print("용량 절약하려면 zip을 /content에 풀지 말고, 한 번만 Drive 쪽에 data/로 풀어두는 것을 권장합니다.")
    else:
        print("[ERROR] Drive에 data/도 없고 data.zip도 없습니다.")

for f in ["open/train.csv", "open/dev.csv", "open/sample_submission.csv"]:
    p = os.path.join(LOCAL_DATA, f)
    ok = "[OK]" if os.path.exists(p) else "[FAIL]"
    print(f"  {ok} {f}")

for split in ["train", "dev", "test"]:
    d = os.path.join(LOCAL_DATA, "open", split)
    if os.path.isdir(d):
        n = len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])
        print(f"  {split}/: {n}개")

free = shutil.disk_usage('/content').free / 1e9
print(f"\n로컬 디스크 여유: {free:.1f} GB")

In [ ]:
# 5. 설정 검증
import os

WORK_DIR  = "/content/dacon"
DRIVE_DIR = "/content/drive/MyDrive/dacon"

print("=" * 50)
print("  환경 검증")
print("=" * 50)

for f in ["models.py", "datasets.py", "train.py", "inference.py"]:
    ok = os.path.exists(os.path.join(WORK_DIR, f))
    print(f"  {'[OK]' if ok else '[FAIL]'} {f}")

tc = os.path.join(WORK_DIR, "data", "open", "train.csv")
print(f"  {'[OK]' if os.path.exists(tc) else '[FAIL]'} train.csv")

ss = os.path.join(WORK_DIR, "data", "shapestacks", "shapestacks", "recordings")
if os.path.exists(ss):
    h6 = [d for d in os.listdir(ss) if "h=6" in d]
    print(f"  [OK] ShapeStacks h=6: {len(h6)} scenarios")
else:
    print(f"  [SKIP] ShapeStacks 없음 (Dacon only)")

ckpt_link = os.path.join(WORK_DIR, "checkpoints")
is_drive = os.path.islink(ckpt_link) and "drive" in os.readlink(ckpt_link)
print(f"  {'[OK]' if is_drive else '[FAIL]'} checkpoints/ -> Drive")

sub_link = os.path.join(WORK_DIR, "submissions")
is_drive2 = os.path.islink(sub_link) and "drive" in os.readlink(sub_link)
print(f"  {'[OK]' if is_drive2 else '[FAIL]'} submissions/ -> Drive")

ckpt_dir = os.path.join(DRIVE_DIR, "checkpoints")
ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pth')]
resume_ckpts = [c for c in ckpts if "ckpt" in c]
best_ckpts   = [c for c in ckpts if "ckpt" not in c]

print()
if resume_ckpts:
    print(f"  Resume 체크포인트: {len(resume_ckpts)}개 -> 자동 이어하기!")
    for c in sorted(resume_ckpts):
        print(f"    {c}")
if best_ckpts:
    print(f"  Best 체크포인트: {len(best_ckpts)}개")
    for c in sorted(best_ckpts):
        print(f"    {c}")
if not ckpts:
    print("  체크포인트 없음 -> 새로 학습 시작")

import shutil
free = shutil.disk_usage('/content').free / 1e9
print(f"\n  디스크 여유: {free:.1f} GB")
print("=" * 50)

In [ ]:
# 6. 학습 설정
# 경쟁 모드 기본값: image-only + dev holdout + 강한 코랩 증강
# 선택: "eva_giant", "dinov3_huge", "siglip_so400m", "eva02_large", "dinov2_large"

BACKBONE = "eva_giant"
PRETRAIN_EPOCHS = 0        # 경쟁 모드에서는 pretrain 생략
FINETUNE_EPOCHS = 70
PATIENCE = 15
INCLUDE_DEV = False        # dev는 leaderboard proxy로 유지
USE_VIDEO = False          # 현재 video fusion은 단순 평균이라 기본 비활성화 권장

import os
os.chdir("/content/dacon")

print(f"백본: {BACKBONE}")
print(f"Pretrain: {PRETRAIN_EPOCHS} ep -> Finetune: {FINETUNE_EPOCHS} ep")
print(f"Patience: {PATIENCE}")
print(f"Dev 포함: {INCLUDE_DEV} | 비디오: {USE_VIDEO}")
print(f"작업 디렉토리: {os.getcwd()}")

pt_path = os.path.join("checkpoints", f"{BACKBONE}_pretrained.pth")
if os.path.exists(pt_path):
    mb = os.path.getsize(pt_path) / 1e6
    print(f"\n[INFO] 사전학습 체크포인트 있음: {pt_path} ({mb:.0f} MB)")
    print("  -> 경쟁 모드에서는 필요 시만 사용")
else:
    print("\n[INFO] ImageNet weight + train-only 5-fold + dev holdout 전략 사용")
    print("  -> Cell 2, 2.5, 3, 5, 6, 8, 10 순서 권장")

In [ ]:
# 7. Stage 1: Pretrain (ShapeStacks h=6 + Dacon)
# 경쟁 모드에서는 기본적으로 건너뜀

import os, sys, shlex, subprocess
os.chdir("/content/dacon")

if PRETRAIN_EPOCHS <= 0:
    print("경쟁 모드 기본값: pretrain 생략")
else:
    cmd = [
        sys.executable, "-u", "train.py",
        "--backbone", BACKBONE,
        "--stage", "pretrain",
        "--pretrain_epochs", str(PRETRAIN_EPOCHS),
        "--batch_size_override", "16",
        "--grad_checkpointing",
        "--resume",
        "--num_workers", "4",
    ]
    cmd_str = shlex.join(cmd)
    print(f"CMD: {cmd_str}\n" + "="*50)

    my_env = os.environ.copy()
    my_env["TQDM_FORCE_TTY"] = "1"

    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=my_env
    )

    while True:
        char = process.stdout.read(1)
        if not char and process.poll() is not None:
            break
        sys.stdout.write(char)
        sys.stdout.flush()

    process.wait()
    if process.returncode != 0:
        print(f"\n[ERROR] Pretrain 실패 (exit code {process.returncode})")

In [ ]:
# 8. Stage 2: Finetune (5-Fold CV)
# 경쟁 모드 기본: image-only + dev holdout + 긴 patience

import os, sys, shlex, subprocess
os.chdir("/content/dacon")

cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--grad_checkpointing",
    "--resume",
    "--num_workers", "4",
]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

if INCLUDE_DEV:
    cmd.append("--include_dev")

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Finetune 실패 (exit code {process.returncode})")

In [ ]:
# (선택) 단일 Fold 테스트 — image-only vs video 비교용

FOLD = 0
QUICK_EPOCHS = 12
COMPARE_VIDEO = False

import os, sys, shlex, subprocess
os.chdir("/content/dacon")

cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(QUICK_EPOCHS),
    "--fold", str(FOLD),
    "--patience", "6",
    "--grad_checkpointing",
    "--resume",
    "--num_workers", "4",
]

if COMPARE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] 실패 (exit code {process.returncode})")

In [ ]:
# 9. 추론 + 제출 파일 생성

import os, sys, shlex, subprocess, glob
import pandas as pd
os.chdir("/content/dacon")

cmd = [
    sys.executable, "-u", "inference.py",
    "--backbones", BACKBONE,
    "--tta",
    "--temperature", "1.0",
]
cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, 
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()

if process.returncode != 0:
    print(f"\n[ERROR] 추론 실패 (exit code {process.returncode})")
else:
    subs = sorted(glob.glob("/content/dacon/submissions/*.csv"))
    if subs:
        latest = subs[-1]
        df = pd.read_csv(latest)
        print(f"\n최신 제출: {os.path.basename(latest)}")
        print(f"  샘플 수: {len(df)}")
        print(f"  unstable_prob: mean={df['unstable_prob'].mean():.4f}, std={df['unstable_prob'].std():.4f}")
        print(df.head())
        print(f"\n제출 파일은 Drive에 저장됨: /content/drive/MyDrive/dacon/submissions/")
    else:
        print("\n제출 파일이 생성되지 않았습니다. 체크포인트가 있는지 확인하세요.")

In [ ]:
# 10. Dev 검증

import os, sys, shlex, subprocess
os.chdir("/content/dacon")

cmd = [
    sys.executable, "-u", "inference.py",
    "--backbones", BACKBONE,
    "--tta",
    "--validate",
]
cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, 
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] 검증 실패 (exit code {process.returncode})")

In [ ]:
# 11. 다중 백본 앙상블 파이프라인
# 경쟁 모드: pretrain 없이 image-only finetune 후 앙상블

BACKBONES = ["eva_giant", "dinov3_huge", "siglip_so400m"]
FINETUNE_EP = 70
PATIENCE = 15
USE_VIDEO_ENSEMBLE = False

import os, sys, subprocess
os.chdir("/content/dacon")

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

def run_cmd(cmd):
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=my_env
    )
    while True:
        char = process.stdout.read(1)
        if not char and process.poll() is not None:
            break
        sys.stdout.write(char)
        sys.stdout.flush()
    process.wait()
    return process.returncode

for bk in BACKBONES:
    print(f"\n{'='*60}")
    print(f"  Finetune: {bk}")
    print(f"{'='*60}")
    cmd = [
        sys.executable, "-u", "train.py",
        "--backbone", bk, "--stage", "finetune",
        "--finetune_epochs", str(FINETUNE_EP),
        "--patience", str(PATIENCE),
        "--grad_checkpointing", "--resume", "--num_workers", "4",
    ]
    if USE_VIDEO_ENSEMBLE:
        cmd += ["--batch_size_override", "4", "--use_video"]
    else:
        cmd += ["--batch_size_override", "16"]
    if run_cmd(cmd) != 0:
        print(f"\n[ERROR] {bk} finetune 실패.")

print(f"\n{'='*60}")
print("  앙상블 추론")
print(f"{'='*60}")
cmd = [
    sys.executable, "-u", "inference.py",
    "--backbones", *BACKBONES,
    "--tta",
]
if run_cmd(cmd) != 0:
    print(f"\n[ERROR] 앙상블 추론 실패.")
else:
    print("\n다중 백본 앙상블 완료!")
    print("제출 파일: /content/drive/MyDrive/dacon/submissions/")